# Programming Language Benchmarks: EDA

Exploratory data analysis of `language_benchmarks.csv` -- a dataset containing **2,200** benchmark measurements across **16 programming languages**, **10 benchmarks**, spanning **2020-2025**.

We examine execution time, memory usage, code verbosity, paradigm effects, typing discipline, and temporal trends.

## Objective

Analyze cross-language benchmark behavior to identify which languages provide the best speed, memory efficiency, and code verbosity trade-offs for common algorithmic workloads.

## Method

1. Load and validate benchmark records.
2. Compare median execution time and memory by language.
3. Evaluate trade-offs by paradigm, typing discipline, and year-over-year trends.
4. Summarize practical recommendations and limitations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

DATA_PATH = Path(".") / "language_benchmarks.csv"
print(f"Data file exists: {DATA_PATH.exists()}")

In [ ]:
# Reproducibility controls
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed set to {SEED}')

## 1. Load Data and Basic Statistics

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(10)

In [ ]:
df.describe()

## 2. Language Distribution

In [ ]:
lang_counts = df["language"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
lang_counts.plot.barh(ax=ax, color=sns.color_palette("viridis", len(lang_counts)))
ax.set_xlabel("Number of benchmark entries")
ax.set_ylabel("Language")
ax.set_title("Benchmark Entry Count by Language")
for i, v in enumerate(lang_counts):
    ax.text(v + 2, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Execution Time Comparison Across Languages

In [ ]:
median_time = df.groupby("language")["execution_time_ms"].median().sort_values()
lang_order = median_time.index.tolist()

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(
    data=df, x="language", y="execution_time_ms",
    order=lang_order, ax=ax, showfliers=False
)
ax.set_yscale("log")
ax.set_ylabel("Execution Time (ms, log scale)")
ax.set_xlabel("Language")
ax.set_title("Execution Time Distribution by Language (outliers hidden)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Median execution time (ms):")
print(median_time.to_string())

## 4. Memory Usage Comparison

In [ ]:
median_mem = df.groupby("language")["memory_usage_mb"].median().sort_values()
mem_order = median_mem.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
sns.boxplot(
    data=df, x="language", y="memory_usage_mb",
    order=mem_order, ax=axes[0], showfliers=False
)
axes[0].set_ylabel("Memory Usage (MB)")
axes[0].set_title("Memory Usage Distribution")
axes[0].tick_params(axis="x", rotation=45)

# Bar plot of medians
median_mem.plot.barh(ax=axes[1], color=sns.color_palette("coolwarm", len(median_mem)))
axes[1].set_xlabel("Median Memory Usage (MB)")
axes[1].set_title("Median Memory Usage by Language")

plt.tight_layout()
plt.show()

## 5. Speed vs Memory Tradeoff

In [ ]:
lang_agg = df.groupby("language").agg(
    median_time=("execution_time_ms", "median"),
    median_mem=("memory_usage_mb", "median"),
    popularity=("popularity_index", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    lang_agg["median_time"], lang_agg["median_mem"],
    s=lang_agg["popularity"] * 30,
    alpha=0.7, edgecolors="k", linewidth=0.5,
    c=range(len(lang_agg)), cmap="tab20"
)
for _, row in lang_agg.iterrows():
    ax.annotate(
        row["language"],
        (row["median_time"], row["median_mem"]),
        textcoords="offset points", xytext=(8, 4), fontsize=9,
    )
ax.set_xlabel("Median Execution Time (ms)")
ax.set_ylabel("Median Memory Usage (MB)")
ax.set_title("Speed vs Memory Tradeoff (bubble size = popularity)")
ax.set_xscale("log")
ax.axvline(lang_agg["median_time"].median(), ls="--", color="grey", alpha=0.5, label="Median of medians")
ax.axhline(lang_agg["median_mem"].median(), ls="--", color="grey", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Lines of Code -- Language Verbosity

In [ ]:
loc_median = df.groupby("language")["lines_of_code"].median().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

loc_median.plot.barh(ax=axes[0], color=sns.color_palette("YlOrRd", len(loc_median)))
axes[0].set_xlabel("Median Lines of Code")
axes[0].set_title("Code Verbosity by Language")

# LOC vs execution time
lang_loc_time = df.groupby("language").agg(
    median_loc=("lines_of_code", "median"),
    median_time=("execution_time_ms", "median"),
).reset_index()
axes[1].scatter(lang_loc_time["median_loc"], lang_loc_time["median_time"], s=80, edgecolors="k")
for _, row in lang_loc_time.iterrows():
    axes[1].annotate(row["language"], (row["median_loc"], row["median_time"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Median Lines of Code")
axes[1].set_ylabel("Median Execution Time (ms)")
axes[1].set_title("Verbosity vs Performance")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 7. Performance by Paradigm

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

paradigm_order = df.groupby("paradigm")["execution_time_ms"].median().sort_values().index

sns.boxplot(data=df, x="paradigm", y="execution_time_ms", order=paradigm_order,
            ax=axes[0], showfliers=False)
axes[0].set_yscale("log")
axes[0].set_title("Execution Time by Paradigm")
axes[0].set_ylabel("Execution Time (ms, log)")

sns.boxplot(data=df, x="paradigm", y="memory_usage_mb", order=paradigm_order,
            ax=axes[1], showfliers=False)
axes[1].set_title("Memory Usage by Paradigm")
axes[1].set_ylabel("Memory (MB)")

sns.boxplot(data=df, x="paradigm", y="lines_of_code", order=paradigm_order,
            ax=axes[2], showfliers=False)
axes[2].set_title("Lines of Code by Paradigm")
axes[2].set_ylabel("Lines of Code")

for ax in axes:
    ax.set_xlabel("Paradigm")

plt.suptitle("Performance Metrics by Programming Paradigm", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print("\nMedian execution time by paradigm:")
print(df.groupby("paradigm")["execution_time_ms"].median().sort_values())

## 8. Static vs Dynamic Typing Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

typing_order = ["static", "gradual", "dynamic"]

sns.violinplot(data=df, x="typing", y="execution_time_ms", order=typing_order,
               ax=axes[0], inner="quartile", cut=0)
axes[0].set_yscale("log")
axes[0].set_title("Execution Time by Typing Discipline")
axes[0].set_ylabel("Execution Time (ms, log)")
axes[0].set_xlabel("Typing")

sns.violinplot(data=df, x="typing", y="memory_usage_mb", order=typing_order,
               ax=axes[1], inner="quartile", cut=0)
axes[1].set_title("Memory Usage by Typing Discipline")
axes[1].set_ylabel("Memory (MB)")
axes[1].set_xlabel("Typing")

plt.suptitle("Static vs Dynamic vs Gradual Typing", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

typing_summary = df.groupby("typing").agg(
    median_time=("execution_time_ms", "median"),
    median_mem=("memory_usage_mb", "median"),
    count=("language", "count"),
    languages=("language", "nunique"),
).sort_values("median_time")
print(typing_summary)

## 9. Temporal Trends (2020-2025)

In [ ]:
yearly = df.groupby(["year", "language"]).agg(
    median_time=("execution_time_ms", "median"),
    median_mem=("memory_usage_mb", "median"),
).reset_index()

# Select a few representative languages
highlight_langs = ["Rust", "C++", "Go", "Python", "JavaScript", "Java"]
yearly_hl = yearly[yearly["language"].isin(highlight_langs)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for lang in highlight_langs:
    subset = yearly_hl[yearly_hl["language"] == lang].sort_values("year")
    axes[0].plot(subset["year"], subset["median_time"], marker="o", label=lang)
    axes[1].plot(subset["year"], subset["median_mem"], marker="s", label=lang)

axes[0].set_title("Median Execution Time Over Years")
axes[0].set_ylabel("Execution Time (ms)")
axes[0].set_xlabel("Year")
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[1].set_title("Median Memory Usage Over Years")
axes[1].set_ylabel("Memory (MB)")
axes[1].set_xlabel("Year")
axes[1].legend(fontsize=8)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Temporal Trends for Selected Languages", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Overall trend
overall_yearly = df.groupby("year")["execution_time_ms"].median()
print("Overall median execution time by year:")
print(overall_yearly)

## 10. Top Performers per Benchmark

In [ ]:
bench_lang = df.groupby(["benchmark_name", "language"]).agg(
    median_time=("execution_time_ms", "median"),
    median_mem=("memory_usage_mb", "median"),
).reset_index()

# Fastest language per benchmark
fastest = bench_lang.loc[bench_lang.groupby("benchmark_name")["median_time"].idxmin()]
print("Fastest language per benchmark (by median time):")
print(fastest[["benchmark_name", "language", "median_time"]].to_string(index=False))
print()

# Heatmap: language vs benchmark
pivot_time = bench_lang.pivot(index="language", columns="benchmark_name", values="median_time")

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    np.log10(pivot_time), annot=pivot_time.round(0).astype(int),
    fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5,
    cbar_kws={"label": "log10(Execution Time ms)"}
)
ax.set_title("Median Execution Time (ms) by Language and Benchmark\n(color = log scale, labels = raw ms)")
ax.set_ylabel("Language")
ax.set_xlabel("Benchmark")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Correlation Heatmap of Numeric Features

In [ ]:
numeric_cols = ["execution_time_ms", "memory_usage_mb", "lines_of_code",
                "cpu_cores", "year", "popularity_index"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=1, ax=ax,
)
ax.set_title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()

print("\nNotable correlations:")
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.3:
            print(f"  {numeric_cols[i]} <-> {numeric_cols[j]}: {r:.3f}")

## 12. Garbage Collection Impact

In [ ]:
gc_order = ["no", "optional", "yes"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="gc", y="execution_time_ms", order=gc_order,
            ax=axes[0], showfliers=False)
axes[0].set_yscale("log")
axes[0].set_title("Execution Time by GC Strategy")
axes[0].set_ylabel("Execution Time (ms, log)")
axes[0].set_xlabel("Garbage Collection")

sns.boxplot(data=df, x="gc", y="memory_usage_mb", order=gc_order,
            ax=axes[1], showfliers=False)
axes[1].set_title("Memory Usage by GC Strategy")
axes[1].set_ylabel("Memory (MB)")
axes[1].set_xlabel("Garbage Collection")

plt.tight_layout()
plt.show()

print(df.groupby("gc").agg(
    median_time=("execution_time_ms", "median"),
    median_mem=("memory_usage_mb", "median"),
    languages=("language", "nunique"),
).loc[gc_order])

## 13. Summary Findings

**Key takeaways from this exploratory analysis:**

1. **Fastest languages:** C++ and Rust consistently sit at the top for raw execution speed, with median times often an order of magnitude lower than interpreted languages.

2. **Memory efficiency:** Rust and C++ again lead in low memory footprint, benefiting from lack of garbage collection overhead. R and Python tend to consume the most memory.

3. **Speed-Memory tradeoff:** There is a visible cluster of systems languages (Rust, C++, Go) in the "fast and lean" quadrant, while interpreted/dynamic languages (Python, R, Ruby) occupy the "slow and heavy" region.

4. **Code verbosity:** Languages like Python and Ruby are the most concise (fewest lines of code), while Java and C++ tend to be more verbose. Interestingly, conciseness does not correlate with performance.

5. **Paradigm effects:** Multi-paradigm languages span the entire performance range. OOP-only languages (Java, C#) cluster in the middle, while functional languages (Haskell, Elixir) show mixed results.

6. **Static vs Dynamic typing:** Statically typed languages are significantly faster on average. Gradually typed languages (TypeScript) sit between the two groups.

7. **Garbage collection:** Languages without GC (Rust, C++) show lower median execution times and memory usage. Optional GC (Swift) sits in between.

8. **Temporal trends:** Performance is relatively stable across 2020-2025 for most languages, suggesting that benchmark improvements are incremental rather than revolutionary.

9. **Benchmark variability:** Performance rankings shift depending on the benchmark (e.g., I/O-bound vs CPU-bound tasks), reinforcing that no single language is universally "fastest."